# Runtime + GPU memory benchmarks (Colab)

Times a few epochs for each of the four training regimes, then linearly extrapolates to a full run. Also reports peak GPU memory.

**Before running:**
1. `Runtime` → `Change runtime type` → select a GPU (T4 / L4 / A100, depending on your Colab tier).
2. Make the project files reachable. Pick **one** setup path in the next cell:
   - **Drive mount** — upload the project folder to your Drive once, then mount.
   - **Zip upload** — use the file pane to upload a zipped copy, then unzip.
   - **Git clone** — if the project is on GitHub.

The real-data experiment (`invPDE_real_4site`) needs `./data/` to be present. Skip it via `experiment_names` below if you only uploaded the training scripts.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# --- Choose ONE setup path; leave the others commented ---

# Option A: mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_DIR = '/content/drive/MyDrive/inverse-turing-testing3'

# Option B: upload a zip via the file pane, then unzip
# !unzip -q /content/inverse-turing-testing3.zip -d /content/
# PROJECT_DIR = '/content/inverse-turing-testing3'

# Option C: git clone
# !git clone <repo-url> /content/inverse-turing-testing3
# PROJECT_DIR = '/content/inverse-turing-testing3'

PROJECT_DIR = '/content/inverse-turing-testing3'  # edit if needed

import os, sys
assert os.path.isdir(PROJECT_DIR), f'Project not found at {PROJECT_DIR}'
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)  # so outputs land here and ./data/ resolves
print('cwd:', os.getcwd())

In [ ]:
import platform
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    device_info = f'CUDA: {torch.cuda.get_device_name(0)}  ({platform.platform()})'
else:
    device_info = f'CPU: {platform.processor() or platform.machine()}'
print(device_info)

In [ ]:
from measure_runtime import measure, TARGET_EPOCHS, Result, write_outputs

# Subset if needed (e.g. drop real-data when ./data/ is absent):
# experiment_names = [n for n in TARGET_EPOCHS if 'real' not in n]
experiment_names = list(TARGET_EPOCHS.keys())

WARMUP = 3
TIMED  = 10  # bump to 20–30 for tighter estimates on the cheaper experiments

results = []
for name in experiment_names:
    try:
        results.append(measure(name, WARMUP, TIMED, device))
    except Exception as e:
        print(f'!! {name} failed: {e}')
        results.append(Result(
            name=name, target_epochs=TARGET_EPOCHS[name],
            timed_epochs=TIMED, setup_seconds=float('nan'),
            mean_epoch_seconds=float('nan'),
            estimated_full_run_seconds=float('nan'),
            device=str(device), notes=f'failed: {e}',
        ))

In [ ]:
from IPython.display import Markdown, display

write_outputs(results, device_info)
with open('runtime_estimates.md') as f:
    display(Markdown(f.read()))